# Results Plots:

### Author : Ricardo Queiroz (UFRN)

In [1]:
import numpy as np
import pandas as pd
import matplotlib.font_manager as fm
import matplotlib.pyplot as plt
from itertools import cycle
import os
import re

font_dir = '/usr/share/fonts/truetype/msttcorefonts/'
fm.fontManager.addfont(font_dir + 'Times_New_Roman.ttf')
plt.rcParams['font.family'] = 'Times New Roman'

## Paths

In [10]:
# Changing Working Directory - Linux
campaigns_path = '../measurement_results/'
campaign_name = 'magazine_paper'
results_path = '../plot_results/'+campaign_name 

#metrics = ['psnr']
metrics = ['psnr', 'ssim', 'tput5g', 'tputFWA', 'vamp']
modulation = ['64qam', '256qam']
users = [1, 3, 5]
#antenna = ['mimo']
antenna = ['siso','mimo']
njobs = 10

In [11]:
dados = {}
acumulados = {}

for n in range(njobs):
    path = os.path.join(campaigns_path, campaign_name, f"JOB{n}/")

    for filename in os.listdir(path):
        complete_path = os.path.join(path, filename)
        var_name = os.path.splitext(filename)[0] + f"_JOB{n}"

        try:
            if filename.endswith('.txt'):
                df_txt = pd.read_csv(complete_path, delimiter=' ', header=None)
                registros = []
                for _, row in df_txt.iterrows():
                    dicionario = {}
                    for item in row.dropna():
                        if isinstance(item, str) and ':' in item:
                            chave, valor = item.split(':', 1)
                            chave = chave.strip()
                            valor = valor.strip()
                            if chave in dicionario:
                                chave = f"{chave}_duplicado"
                            dicionario[chave] = valor
                    registros.append(dicionario)
                df= pd.DataFrame(registros)
                for col in df.columns:
                    try:
                        df[col] = pd.to_numeric(df[col])
                    except (ValueError, TypeError):
                        pass
            
            elif filename.endswith('.csv'):
                # Verifica se o conteúdo tem padrão chave:valor como no antigo txt
                with open(complete_path, 'r') as f:
                    primeira_linha = f.readline()
                
                if ':' in primeira_linha and ',' not in primeira_linha:
                    # Trata como antigo txt, mesmo sendo .csv
                    df_txt = pd.read_csv(complete_path, delimiter=' ', header=None)
                    registros = []
                    for _, row in df_txt.iterrows():
                        dicionario = {}
                        for item in row.dropna():
                            if isinstance(item, str) and ':' in item:
                                chave, valor = item.split(':', 1)
                                chave = chave.strip()
                                valor = valor.strip()
                                if chave in dicionario:
                                    chave = f"{chave}_duplicado"
                                dicionario[chave] = valor
                        registros.append(dicionario)
                    df = pd.DataFrame(registros)
                    for col in df.columns:
                        try:
                            df[col] = pd.to_numeric(df[col])
                        except (ValueError, TypeError):
                            pass
                else:
                    # Trata como CSV padrão com colunas separadas por vírgula
                    df_csv = pd.read_csv(complete_path, delimiter=',')
            
                    bitrate_cols = [col for col in df_csv.columns if
                        df_csv[col].astype(str).str.contains(r'(?:b/s|kb/s|Mb/s)', regex=True).any()]
            
                    def convert_to_mbps(val):
                        if isinstance(val, str):
                            val = val.strip()
                            try:
                                if val.endswith('Mb/s'):
                                    return float(val.replace(' Mb/s', '').replace('Mb/s', ''))
                                elif val.endswith('kb/s'):
                                    return float(val.replace(' kb/s', '').replace('kb/s', '')) / 1000
                                elif val.endswith('b/s'):
                                    return float(val.replace(' b/s', '').replace('b/s', '')) / 1_000_000
                                else:
                                    return float(val)
                            except:
                                return None
                        return val
            
                    for col in bitrate_cols:
                        df_csv[col] = df_csv[col].apply(convert_to_mbps)
                        df_csv.rename(columns={col: f"{col} (Mbps)"}, inplace=True)
            
                    df = pd.DataFrame(df_csv)
            
                    if df.shape[1] >= 3:
                        df.rename(columns={
                            df.columns[0]: 'tempo_s',
                            df.columns[1]: 'downlink_Mbps',
                            df.columns[2]: 'uplink_Mbps'
                        }, inplace=True)
            
                        df['tempo_s'] = pd.to_datetime(df['tempo_s'], errors='coerce')
                        df = df.dropna(subset=['tempo_s'])
            
                        mask_nonzero = ~(df['downlink_Mbps'] == 0)
                        nonzero_indices = df[mask_nonzero].index
            
                        if not nonzero_indices.empty:
                            first_valid_idx = nonzero_indices[0]
                            last_valid_idx = nonzero_indices[-1]
                            df = df.iloc[first_valid_idx:last_valid_idx + 1].reset_index(drop=True)
            
                        t_inicial = df['tempo_s'].iloc[0]
                        df['tempo_s'] = (df['tempo_s'] - t_inicial).dt.total_seconds()
            
            elif filename.endswith('.xml'):
                df = pd.read_xml(complete_path, xpath=".//frame")
            else:
                continue
            dados[var_name] = df
        except Exception as e:
            print(f"Erro ao ler {filename}: {e}")

for var_name, df in dados.items():
    if var_name not in acumulados:
        acumulados[var_name] = []
    acumulados[var_name].append(df)

for var_name in acumulados:
    acumulados[var_name] = pd.concat(acumulados[var_name])

Erro ao ler quality_256qam_siso_5user.csv: single positional indexer is out-of-bounds
Erro ao ler quality_64qam_siso_1user.csv: single positional indexer is out-of-bounds


In [120]:
grupo_dfs = {
    'psnr_256qam_siso_1user': [],
    'psnr_256qam_siso_3user': [],
    'psnr_256qam_siso_5user': [],
    'psnr_64qam_siso_1user': [],
    'psnr_64qam_siso_3user': [],
    'psnr_64qam_siso_5user': [],
    'vmaf_256qam_siso_1user': [],
    'vmaf_256qam_siso_3user': [],
    'vmaf_256qam_siso_5user': [],
    'vmaf_64qam_siso_1user': [],
    'vmaf_64qam_siso_3user': [],
    'vmaf_64qam_siso_5user': [],
    'tput5g_256qam_siso_1user': [],
    'tput5g_256qam_siso_3user': [],
    'tput5g_256qam_siso_5user': [],
    'tput5g_64qam_siso_1user': [],
    'tput5g_64qam_siso_3user': [],
    'tput5g_64qam_siso_5user': [],
    'tputFWA_256qam_siso_1user': [],
    'tputFWA_256qam_siso_3user': [],
    'tputFWA_256qam_siso_5user': [],
    'tputFWA_64qam_siso_1user': [],
    'tputFWA_64qam_siso_3user': [],
    'tputFWA_64qam_siso_5user': [],
    'ssim_256qam_siso_1user': [],
    'ssim_256qam_siso_3user': [],
    'ssim_256qam_siso_5user': [],
    'ssim_64qam_siso_1user': [],
    'ssim_64qam_siso_3user': [],
    'ssim_64qam_siso_5user': [],
    
    'psnr_256qam_mimo_1user': [],
    'psnr_256qam_mimo_3user': [],
    'psnr_256qam_mimo_5user': [],
    'psnr_64qam_mimo_1user': [],
    'psnr_64qam_mimo_3user': [],
    'psnr_64qam_mimo_5user': [],
    'vmaf_256qam_mimo_1user': [],
    'vmaf_256qam_mimo_3user': [],
    'vmaf_256qam_mimo_5user': [],
    'vmaf_64qam_mimo_1user': [],
    'vmaf_64qam_mimo_3user': [],
    'vmaf_64qam_mimo_5user': [],
    'tput5g_256qam_mimo_1user': [],
    'tput5g_256qam_mimo_3user': [],
    'tput5g_256qam_mimo_5user': [],
    'tput5g_64qam_mimo_1user': [],
    'tput5g_64qam_mimo_3user': [],
    'tput5g_64qam_mimo_5user': [],
    'tputFWA_256qam_mimo_1user': [],
    'tputFWA_256qam_mimo_3user': [],
    'tputFWA_256qam_mimo_5user': [],
    'tputFWA_64qam_mimo_1user': [],
    'tputFWA_64qam_mimo_3user': [],
    'tputFWA_64qam_mimo_5user': [],
    'ssim_256qam_mimo_1user': [],
    'ssim_256qam_mimo_3user': [],
    'ssim_256qam_mimo_5user': [],
    'ssim_64qam_mimo_1user': [],
    'ssim_64qam_mimo_3user': [],
    'ssim_64qam_mimo_5user': []
}

for k in acumulados:
    if re.match(r'psnr_256qam_siso_1user_JOB\d+', k):
        grupo_dfs['psnr_256qam_siso_1user'].append(acumulados[k])
    elif re.match(r'psnr_256qam_siso_3user_JOB\d+', k):
        grupo_dfs['psnr_256qam_siso_3user'].append(acumulados[k])
    elif re.match(r'psnr_256qam_siso_5user_JOB\d+', k):
        grupo_dfs['psnr_256qam_siso_5user'].append(acumulados[k])
    elif re.match(r'psnr_64qam_siso_1user_JOB\d+', k):
        grupo_dfs['psnr_64qam_siso_1user'].append(acumulados[k])
    elif re.match(r'psnr_64qam_siso_3user_JOB\d+', k):
        grupo_dfs['psnr_64qam_siso_3user'].append(acumulados[k])
    elif re.match(r'psnr_64qam_siso_5user_JOB\d+', k):
        grupo_dfs['psnr_64qam_siso_5user'].append(acumulados[k])
    elif re.match(r'vmaf_256qam_siso_1user_JOB\d+', k):
        grupo_dfs['vmaf_256qam_siso_1user'].append(acumulados[k])
    elif re.match(r'vmaf_256qam_siso_3user_JOB\d+', k):
        grupo_dfs['vmaf_256qam_siso_3user'].append(acumulados[k])
    elif re.match(r'vmaf_256qam_siso_5user_JOB\d+', k):
        grupo_dfs['vmaf_256qam_siso_5user'].append(acumulados[k])
    elif re.match(r'vmaf_64qam_siso_1user_JOB\d+', k):
        grupo_dfs['vmaf_64qam_siso_1user'].append(acumulados[k])
    elif re.match(r'vmaf_64qam_siso_3user_JOB\d+', k):
        grupo_dfs['vmaf_64qam_siso_3user'].append(acumulados[k])
    elif re.match(r'vmaf_64qam_siso_5user_JOB\d+', k):
        grupo_dfs['vmaf_64qam_siso_5user'].append(acumulados[k])
    elif re.match(r'ssim_256qam_siso_1user_JOB\d+', k):
        grupo_dfs['ssim_256qam_siso_1user'].append(acumulados[k])
    elif re.match(r'ssim_256qam_siso_3user_JOB\d+', k):
        grupo_dfs['ssim_256qam_siso_3user'].append(acumulados[k])
    elif re.match(r'ssim_256qam_siso_5user_JOB\d+', k):
        grupo_dfs['ssim_256qam_siso_5user'].append(acumulados[k])
    elif re.match(r'ssim_64qam_siso_1user_JOB\d+', k):
        grupo_dfs['ssim_64qam_siso_1user'].append(acumulados[k])
    elif re.match(r'ssim_64qam_siso_3user_JOB\d+', k):
        grupo_dfs['ssim_64qam_siso_3user'].append(acumulados[k])
    elif re.match(r'ssim_64qam_siso_5user_JOB\d+', k):
        grupo_dfs['ssim_64qam_siso_5user'].append(acumulados[k])
    elif re.match(r'tput5g_256qam_siso_1user_JOB\d+', k):
        grupo_dfs['tput5g_256qam_siso_1user'].append(acumulados[k])
    elif re.match(r'tput5g_256qam_siso_3user_JOB\d+', k):
        grupo_dfs['tput5g_256qam_siso_3user'].append(acumulados[k])
    elif re.match(r'tput5g_256qam_siso_5user_JOB\d+', k):
        grupo_dfs['tput5g_256qam_siso_5user'].append(acumulados[k])
    elif re.match(r'tput5g_64qam_siso_1user_JOB\d+', k):
        grupo_dfs['tput5g_64qam_siso_1user'].append(acumulados[k])
    elif re.match(r'tput5g_64qam_siso_3user_JOB\d+', k):
        grupo_dfs['tput5g_64qam_siso_3user'].append(acumulados[k])
    elif re.match(r'tput5g_64qam_siso_5user_JOB\d+', k):
        grupo_dfs['tput5g_64qam_siso_5user'].append(acumulados[k])
    elif re.match(r'tputFWA_256qam_siso_1user_JOB\d+', k):
        grupo_dfs['tputFWA_256qam_siso_1user'].append(acumulados[k])
    elif re.match(r'tputFWA_256qam_siso_3user_JOB\d+', k):
        grupo_dfs['tputFWA_256qam_siso_3user'].append(acumulados[k])
    elif re.match(r'tputFWA_256qam_siso_5user_JOB\d+', k):
        grupo_dfs['tputFWA_256qam_siso_5user'].append(acumulados[k])
    elif re.match(r'tputFWA_64qam_siso_1user_JOB\d+', k):
        grupo_dfs['tputFWA_64qam_siso_1user'].append(acumulados[k])
    elif re.match(r'tputFWA_64qam_siso_3user_JOB\d+', k):
        grupo_dfs['tputFWA_64qam_siso_3user'].append(acumulados[k])
    elif re.match(r'tputFWA_64qam_siso_5user_JOB\d+', k):
        grupo_dfs['tputFWA_64qam_siso_5user'].append(acumulados[k])
    
    elif re.match(r'psnr_256qam_mimo_1user_JOB\d+', k):
        grupo_dfs['psnr_256qam_mimo_1user'].append(acumulados[k])
    elif re.match(r'psnr_256qam_mimo_3user_JOB\d+', k):
        grupo_dfs['psnr_256qam_mimo_3user'].append(acumulados[k])
    elif re.match(r'psnr_256qam_mimo_5user_JOB\d+', k):
        grupo_dfs['psnr_256qam_mimo_5user'].append(acumulados[k])
    elif re.match(r'psnr_64qam_mimo_1user_JOB\d+', k):
        grupo_dfs['psnr_64qam_mimo_1user'].append(acumulados[k])
    elif re.match(r'psnr_64qam_mimo_3user_JOB\d+', k):
        grupo_dfs['psnr_64qam_mimo_3user'].append(acumulados[k])
    elif re.match(r'psnr_64qam_mimo_5user_JOB\d+', k):
        grupo_dfs['psnr_64qam_mimo_5user'].append(acumulados[k])
    elif re.match(r'vmaf_256qam_mimo_1user_JOB\d+', k):
        grupo_dfs['vmaf_256qam_mimo_1user'].append(acumulados[k])
    elif re.match(r'vmaf_256qam_mimo_3user_JOB\d+', k):
        grupo_dfs['vmaf_256qam_mimo_3user'].append(acumulados[k])
    elif re.match(r'vmaf_256qam_mimo_5user_JOB\d+', k):
        grupo_dfs['vmaf_256qam_mimo_5user'].append(acumulados[k])
    elif re.match(r'vmaf_64qam_mimo_1user_JOB\d+', k):
        grupo_dfs['vmaf_64qam_mimo_1user'].append(acumulados[k])
    elif re.match(r'vmaf_64qam_mimo_3user_JOB\d+', k):
        grupo_dfs['vmaf_64qam_mimo_3user'].append(acumulados[k])
    elif re.match(r'vmaf_64qam_mimo_5user_JOB\d+', k):
        grupo_dfs['vmaf_64qam_mimo_5user'].append(acumulados[k])
    elif re.match(r'ssim_256qam_mimo_1user_JOB\d+', k):
        grupo_dfs['ssim_256qam_mimo_1user'].append(acumulados[k])
    elif re.match(r'ssim_256qam_mimo_3user_JOB\d+', k):
        grupo_dfs['ssim_256qam_mimo_3user'].append(acumulados[k])
    elif re.match(r'ssim_256qam_mimo_5user_JOB\d+', k):
        grupo_dfs['ssim_256qam_mimo_5user'].append(acumulados[k])
    elif re.match(r'ssim_64qam_mimo_1user_JOB\d+', k):
        grupo_dfs['ssim_64qam_mimo_1user'].append(acumulados[k])
    elif re.match(r'ssim_64qam_mimo_3user_JOB\d+', k):
        grupo_dfs['ssim_64qam_mimo_3user'].append(acumulados[k])
    elif re.match(r'ssim_64qam_mimo_5user_JOB\d+', k):
        grupo_dfs['ssim_64qam_mimo_5user'].append(acumulados[k])
    elif re.match(r'tput5g_256qam_mimo_1user_JOB\d+', k):
        grupo_dfs['tput5g_256qam_mimo_1user'].append(acumulados[k])
    elif re.match(r'tput5g_256qam_mimo_3user_JOB\d+', k):
        grupo_dfs['tput5g_256qam_mimo_3user'].append(acumulados[k])
    elif re.match(r'tput5g_256qam_mimo_5user_JOB\d+', k):
        grupo_dfs['tput5g_256qam_mimo_5user'].append(acumulados[k])
    elif re.match(r'tput5g_64qam_mimo_1user_JOB\d+', k):
        grupo_dfs['tput5g_64qam_mimo_1user'].append(acumulados[k])
    elif re.match(r'tput5g_64qam_mimo_3user_JOB\d+', k):
        grupo_dfs['tput5g_64qam_mimo_3user'].append(acumulados[k])
    elif re.match(r'tput5g_64qam_mimo_5user_JOB\d+', k):
        grupo_dfs['tput5g_64qam_mimo_5user'].append(acumulados[k])
    elif re.match(r'tputFWA_256qam_mimo_1user_JOB\d+', k):
        grupo_dfs['tputFWA_256qam_mimo_1user'].append(acumulados[k])
    elif re.match(r'tputFWA_256qam_mimo_3user_JOB\d+', k):
        grupo_dfs['tputFWA_256qam_mimo_3user'].append(acumulados[k])
    elif re.match(r'tputFWA_256qam_mimo_5user_JOB\d+', k):
        grupo_dfs['tputFWA_256qam_mimo_5user'].append(acumulados[k])
    elif re.match(r'tputFWA_64qam_mimo_1user_JOB\d+', k):
        grupo_dfs['tputFWA_64qam_mimo_1user'].append(acumulados[k])
    elif re.match(r'tputFWA_64qam_mimo_3user_JOB\d+', k):
        grupo_dfs['tputFWA_64qam_mimo_3user'].append(acumulados[k])
    elif re.match(r'tputFWA_64qam_mimo_5user_JOB\d+', k):
        grupo_dfs['tputFWA_64qam_mimo_5user'].append(acumulados[k])

# Agora calcular o empilhamento e a média para cada grupo
medias_por_job = {}

for grupo, dfs in grupo_dfs.items():
    if dfs:  # evita erro se a lista estiver vazia
        stacked = pd.concat(dfs, keys=range(len(dfs)), axis=0)
        if str(grupo).startswith('psnr_'):
            # Seleciona apenas colunas PSNR
            psnr_cols = [c for c in stacked.columns if 'psnr' in c.lower()]
            # Copia o dataframe para não alterar o original
            stacked_linear = stacked.copy()
            # Converte colunas PSNR de dB → linear
            stacked_linear[psnr_cols] = 10 ** (stacked_linear[psnr_cols] / 10)
            # Faz a média em linear
            media_linear = stacked_linear.groupby(level=1).mean()
            # Converte de volta para dB
            media_linear[psnr_cols] = 10 * np.log10(media_linear[psnr_cols])
            # Agora sim, média em dB corretamente
            medias_por_job[grupo] = media_linear
        else:
            # Caso normal
            medias_por_job[grupo] = stacked.groupby(level=1).mean()

## Plots

In [99]:
fontSize = 24
fontSizeLeg = 20
figsize=(10,8)
markers = ['o', 's', '^', 'D', 'v', '*', 'x', 'P', 'H', '<', '>']
hatch=['x','+','o']
cmap = plt.get_cmap("tab10")
plt.rc('font', size=fontSize)

### PSNR

In [106]:
y_linear.size

10800

In [171]:
for mode in antenna:
    for user in users:
        key_64qam = f'psnr_64qam_{mode}_{user}user'
        key_256qam = f'psnr_256qam_{mode}_{user}user'
    
        # Verificações de existência dos dados
        if key_64qam not in medias_por_job or key_256qam not in medias_por_job:
            print(f"Dados ausentes para {user} usuário(s). Pulando.")
            continue
    
        # Conversão dos dados
        x = medias_por_job[key_64qam]['n'].astype(float).values
        y = medias_por_job[key_64qam]['psnr_avg'].astype(float).values
        x2 = medias_por_job[key_256qam]['n'].astype(float).values
        y2 = medias_por_job[key_256qam]['psnr_avg'].astype(float).values
    
        # Parâmetros
        limiar = 0.1
        min_consecutivos = 3
        
        # Encontrar o índice onde começam os valores consecutivos abaixo do limiar
        for i in range(len(y) - min_consecutivos + 1):
            if np.all(y[i:i + min_consecutivos] < limiar):
                corte = i
                break
        else:
            corte = len(y)  # Não encontrou sequência, usa o vetor todo
    
        for i in range(len(y2) - min_consecutivos + 1):
            if np.all(y2[i:i + min_consecutivos] < limiar):
                corte2 = i
                break
        else:
            corte2 = len(y2)  # Não encontrou sequência, usa o vetor todo
        
        
        # Calcular a média até o ponto de corte
        #print(f'Corte 1: {corte}')
        #print(f'Corte 2: {corte2}')
        # Converte de dB para linear
        y_linear = 10 ** (y[:corte] / 10)
        y2_linear = 10 ** (y2[:corte2] / 10)
        
        # Calcula a média no domínio linear
        mean_y_linear = y_linear[y_linear < 1000000].mean()
        mean_y2_linear = y2_linear[y2_linear < 1000000].mean()
        
        # Converte de volta para dB
        mean_y = 10 * np.log10(mean_y_linear)
        mean_y2 = 10 * np.log10(mean_y2_linear)
        # Plot
        plt.figure(figsize=figsize)
        plt.plot(x, y, linestyle='-', linewidth=1.5, color=cmap(0), label='MCS$_{\mathrm{MAX}}$ com 64-QAM')
        plt.plot(x2, y2, linestyle='-', linewidth=1.5, color=cmap(3), label='MCS$_{\mathrm{MAX}}$ com 256-QAM')
    
        # Linhas horizontais das médias
        plt.axhline(mean_y, color=cmap(0), linestyle='--', linewidth=1.5, label=f'Média 64-QAM: {mean_y:.2f} dB')
        plt.axhline(mean_y2, color=cmap(3), linestyle='--', linewidth=1.5, label=f'Média 256-QAM: {mean_y2:.2f} dB')
    
        # Ajustes do gráfico
        #plt.title(f'PSNR - {user} User(s) - {mode}')
        plt.xlabel('Frames', fontsize=fontSize)
        plt.ylabel('PSNR (dB)', fontsize=fontSize)
        plt.grid(True, linestyle='--', linewidth=0.7, alpha=0.6)
        plt.xlim(0,10800)
        plt.ylim(25,60)
        plt.legend(fontsize=fontSizeLeg)
        plt.tight_layout()
    
        # Salvar figura
        os.makedirs(results_path, exist_ok=True)
        complete_path = os.path.join(results_path, f"psnr_{mode}_{user}user.png")
        plt.savefig(complete_path, dpi=600, bbox_inches='tight', transparent=True)
        plt.close()

### SSIM

In [45]:
for mode in antenna:
    for user in users:
        key_64qam = f'ssim_64qam_{mode}_{user}user'
        key_256qam = f'ssim_256qam_{mode}_{user}user'
    
        # Verificações de existência dos dados
        if key_64qam not in medias_por_job or key_256qam not in medias_por_job:
            print(f"Dados ausentes para {user} usuário(s). Pulando.")
            continue
    
        # Conversão dos dados
        x = medias_por_job[key_64qam]['n'].astype(float).values
        y = medias_por_job[key_64qam]['All'].astype(float).values
        x2 = medias_por_job[key_256qam]['n'].astype(float).values
        y2 = medias_por_job[key_256qam]['All'].astype(float).values

        mean_y = y.mean()
        mean_y2 = y2.mean()
        # Plot
        plt.figure(figsize=figsize)
        plt.plot(x, y, linestyle='-', linewidth=1.5, color=cmap(0), label='MCS$_{\mathrm{MAX}}$ with 64-QAM')
        plt.plot(x2, y2, linestyle='-', linewidth=1.5, color=cmap(3), label='MCS$_{\mathrm{MAX}}$ with 256-QAM')
    
        # Linhas horizontais das médias
        plt.axhline(mean_y, color=cmap(0), linestyle='--', linewidth=1.5, label=f'64-QAM Mean: {mean_y:.2f}')
        plt.axhline(mean_y2, color=cmap(3), linestyle='--', linewidth=1.5, label=f'256-QAM Mean: {mean_y2:.2f}')
    
        # Ajustes do gráfico
        #plt.title(f'SSIM - {user} User(s) - {mode}')
        plt.xlabel('Frames', fontsize=fontSize)
        plt.ylabel('SSIM', fontsize=fontSize)
        plt.grid(True, linestyle='--', linewidth=0.7, alpha=0.6)
        plt.xlim(0,10800)
        if mode == 'siso':
            plt.ylim(bottom=0.70)
        else: 
            plt.ylim(bottom=0.90)
        plt.legend(fontsize=fontSizeLeg)
        plt.tight_layout()
    
        # Salvar figura
        os.makedirs(results_path, exist_ok=True)
        complete_path = os.path.join(results_path, f"ssim_{mode}_{user}user.png")
        plt.savefig(complete_path, dpi=600, bbox_inches='tight', transparent=True)
        plt.close()

### VAMF

In [8]:
for mode in antenna:
    for user in users:
        key_64qam = f'vmaf_64qam_{mode}_{user}user'
        key_256qam = f'vmaf_256qam_{mode}_{user}user'
    
        # Verificações de existência dos dados
        if key_64qam not in medias_por_job or key_256qam not in medias_por_job:
            print(f"Dados ausentes para {user} usuário(s). Pulando.")
            continue
    
        # Conversão dos dados
        x = medias_por_job[key_64qam]['frameNum'].astype(float).values
        y = medias_por_job[key_64qam]['vmaf'].astype(float).values
        x2 = medias_por_job[key_256qam]['frameNum'].astype(float).values
        y2 = medias_por_job[key_256qam]['vmaf'].astype(float).values
    
        # Parâmetros
        limiar = 0.1
        min_consecutivos = 3
        
        # Encontrar o índice onde começam os valores consecutivos abaixo do limiar
        for i in range(len(y) - min_consecutivos + 1):
            if np.all(y[i:i + min_consecutivos] < limiar):
                corte = i
                break
        else:
            corte = len(y)  # Não encontrou sequência, usa o vetor todo
    
        for i in range(len(y2) - min_consecutivos + 1):
            if np.all(y2[i:i + min_consecutivos] < limiar):
                corte2 = i
                break
        else:
            corte2 = len(y2)  # Não encontrou sequência, usa o vetor todo
        
        
        # Calcular a média até o ponto de corte
        #print(f'Corte 1: {corte}')
        #print(f'Corte 2: {corte2}')
        mean_y = y[:corte].mean()
        mean_y2 = y2[:corte2].mean()
        # Plot
        plt.figure(figsize=figsize)
        plt.plot(x, y, linestyle='-', linewidth=1.5, color=cmap(0), label='MCS$_{\mathrm{MAX}}$ with 64-QAM')
        plt.plot(x2, y2, linestyle='-', linewidth=1.5, color=cmap(3), label='MCS$_{\mathrm{MAX}}$ with 256-QAM')
    
        # Linhas horizontais das médias
        plt.axhline(mean_y, color=cmap(0), linestyle='--', linewidth=1.5, label=f'64-QAM Mean: {mean_y:.2f} %')
        plt.axhline(mean_y2, color=cmap(3), linestyle='--', linewidth=1.5, label=f'256-QAM Mean: {mean_y2:.2f} %')
    
        # Ajustes do gráfico
        #plt.title(f'VMAF - {user} User(s) - {mode}')
        plt.xlabel('Frames', fontsize=fontSize)
        plt.ylabel('VMAF (%)', fontsize=fontSize)
        plt.grid(True, linestyle='--', linewidth=0.7, alpha=0.6)
        plt.xlim(0,10800)
        if mode == 'siso':
            plt.ylim(bottom= 50, top=100)
        else: 
            plt.ylim(bottom= 80, top=100)
        plt.legend(fontsize=fontSizeLeg)
        plt.tight_layout()
    
        # Salvar figura
        os.makedirs(results_path, exist_ok=True)
        complete_path = os.path.join(results_path, f"vmaf_{mode}_{user}user.png")
        plt.savefig(complete_path, dpi=600, bbox_inches='tight', transparent=True)
        plt.close()

### Tput

In [9]:
for mode in antenna:
    for user in users:
        key_64qam = f'tputFWA_64qam_{mode}_{user}user'
        key_256qam = f'tputFWA_256qam_{mode}_{user}user'
    
        # Verificações de existência dos dados
        if key_64qam not in medias_por_job or key_256qam not in medias_por_job:
            print(f"Dados ausentes para {user} usuário(s). Pulando.")
            continue
    
        # Conversão dos dados
        x = medias_por_job[key_64qam]['tempo_s'].astype(float).values
        y = medias_por_job[key_64qam]['downlink_Mbps'].astype(float).values
        x2 = medias_por_job[key_256qam]['tempo_s'].astype(float).values
        y2 = medias_por_job[key_256qam]['downlink_Mbps'].astype(float).values
    
        # Parâmetros
        limiar = 0.1
        min_consecutivos = 3
        
        # Encontrar o índice onde começam os valores consecutivos abaixo do limiar
        for i in range(len(y) - min_consecutivos + 1):
            if np.all(y[i:i + min_consecutivos] < limiar):
                corte = i
                break
        else:
            corte = len(y)  # Não encontrou sequência, usa o vetor todo
    
        for i in range(len(y2) - min_consecutivos + 1):
            if np.all(y2[i:i + min_consecutivos] < limiar):
                corte2 = i
                break
        else:
            corte2 = len(y2)  # Não encontrou sequência, usa o vetor todo
        
        
        # Calcular a média até o ponto de corte
        #print(f'Corte 1: {corte}')
        #print(f'Corte 2: {corte2}')
        mean_y = y[:corte].mean()
        mean_y2 = y2[:corte2].mean()
        # Plot
        plt.figure(figsize=figsize)
        plt.plot(x, y, linestyle='-', linewidth=1.5, color=cmap(0), label='MCS$_{\mathrm{MAX}}$ with 64-QAM')
        plt.plot(x2, y2, linestyle='-', linewidth=1.5, color=cmap(3), label='MCS$_{\mathrm{MAX}}$ with 256-QAM')
    
        # Linhas horizontais das médias
        plt.axhline(mean_y, color=cmap(0), linestyle='--', linewidth=1.5, label=f'64-QAM Mean: {mean_y:.2f} Mbps')
        plt.axhline(mean_y2, color=cmap(3), linestyle='--', linewidth=1.5, label=f'256-QAM Mean: {mean_y2:.2f} Mbps')
    
        # Ajustes do gráfico
        #plt.title(f'Throughput WiFi - {user} User(s) - {mode}')
        plt.xlabel('Time (s)', fontsize=fontSize)
        plt.ylabel('Taxa Útil (Mbps)', fontsize=fontSize)
        plt.grid(True, linestyle='--', linewidth=0.7, alpha=0.6)
        plt.xlim(0,180)
        plt.ylim(bottom=0)
        plt.legend(fontsize=fontSizeLeg)
        plt.tight_layout()
    
        # Salvar figura
        os.makedirs(results_path, exist_ok=True)
        complete_path = os.path.join(results_path, f"tput_WiFi_{mode}_{user}user.png")
        plt.savefig(complete_path, dpi=600, bbox_inches='tight', transparent=True)
        plt.close()

In [10]:
for mode in antenna:
    for user in users:
        key_64qam = f'tput5g_64qam_{mode}_{user}user'
        key_256qam = f'tput5g_256qam_{mode}_{user}user'
    
        # Verificações de existência dos dados
        if key_64qam not in medias_por_job or key_256qam not in medias_por_job:
            print(f"Dados ausentes para {user} usuário(s). Pulando.")
            continue
    
        # Conversão dos dados
        x = medias_por_job[key_64qam]['tempo_s'].astype(float).values
        y = medias_por_job[key_64qam]['downlink_Mbps'].astype(float).values
        x2 = medias_por_job[key_256qam]['tempo_s'].astype(float).values
        y2 = medias_por_job[key_256qam]['downlink_Mbps'].astype(float).values
    
        # Parâmetros
        limiar = 0.1
        min_consecutivos = 3
        
        # Encontrar o índice onde começam os valores consecutivos abaixo do limiar
        for i in range(len(y) - min_consecutivos + 1):
            if np.all(y[i:i + min_consecutivos] < limiar):
                corte = i
                break
        else:
            corte = len(y)  # Não encontrou sequência, usa o vetor todo
    
        for i in range(len(y2) - min_consecutivos + 1):
            if np.all(y2[i:i + min_consecutivos] < limiar):
                corte2 = i
                break
        else:
            corte2 = len(y2)  # Não encontrou sequência, usa o vetor todo
        
        
        # Calcular a média até o ponto de corte
        #print(f'Corte 1: {corte}')
        #print(f'Corte 2: {corte2}')
        mean_y = y[:corte].mean()
        mean_y2 = y2[:corte2].mean()
    
        # Plot
        plt.figure(figsize=figsize)
        plt.plot(x, y, linestyle='-', linewidth=1.5, color=cmap(0), label='MCS$_{\mathrm{MAX}}$ with 64-QAM')
        plt.plot(x2, y2, linestyle='-', linewidth=1.5, color=cmap(3), label='MCS$_{\mathrm{MAX}}$ with 256-QAM')
    
        # Linhas horizontais das médias
        plt.axhline(mean_y, color=cmap(0), linestyle='--', linewidth=1.5, label=f'64-QAM Mean: {mean_y:.2f} Mbps')
        plt.axhline(mean_y2, color=cmap(3), linestyle='--', linewidth=1.5, label=f'256-QAM Mean: {mean_y2:.2f} Mbps')
    
        # Ajustes do gráfico
        #plt.title(f'Throughput 5G - {user} User(s) - {mode}')
        plt.xlabel('Time (s)', fontsize=fontSize)
        plt.ylabel('Throughput (Mbps)', fontsize=fontSize)
        plt.grid(True, linestyle='--', linewidth=0.7, alpha=0.6)
        plt.xlim(0,180) 
        if mode == 'siso':
            plt.ylim(bottom= 0, top=24)
        else: 
            plt.ylim(bottom= 0, top=101)
        plt.legend(fontsize=fontSizeLeg)
        plt.tight_layout()
    
        # Salvar figura
        os.makedirs(results_path, exist_ok=True)
        complete_path = os.path.join(results_path, f"tput_5g_{mode}_{user}user.png")
        plt.savefig(complete_path, dpi=600, bbox_inches='tight', transparent=True)
        plt.close()

#### CDF

In [11]:
from matplotlib.lines import Line2D

# Armazenar dados (sem offset)
cdf_data_64qam = []
cdf_data_256qam = []

for user in users:
    # SISO
    key_64qam_siso = f'tputFWA_64qam_siso_{user}user'
    key_256qam_siso = f'tputFWA_256qam_siso_{user}user'

    # MIMO
    key_64qam_mimo = f'tputFWA_64qam_mimo_{user}user'
    key_256qam_mimo = f'tputFWA_256qam_mimo_{user}user'

    # Verificação
    keys_required = [key_64qam_siso, key_256qam_siso, key_64qam_mimo, key_256qam_mimo]
    if any(k not in medias_por_job for k in keys_required):
        print(f"Dados ausentes para {user} usuário(s). Pulando.")
        continue

    # Coleta dos dados
    y_siso_64 = medias_por_job[key_64qam_siso]['downlink_Mbps'].astype(float).values
    y_siso_256 = medias_por_job[key_256qam_siso]['downlink_Mbps'].astype(float).values
    y_mimo_64 = medias_por_job[key_64qam_mimo]['downlink_Mbps'].astype(float).values
    y_mimo_256 = medias_por_job[key_256qam_mimo]['downlink_Mbps'].astype(float).values

    # Armazenamento
    cdf_data_64qam.append((f'{user} User(s) - SISO', y_siso_64))
    cdf_data_64qam.append((f'{user} User(s) - MIMO', y_mimo_64))
    cdf_data_256qam.append((f'{user} User(s) - SISO', y_siso_256))
    cdf_data_256qam.append((f'{user} User(s) - MIMO', y_mimo_256))

# --- Plot CDF ---
fig, ax = plt.subplots(figsize=figsize)

legend_handles = []

# 64-QAM: curva contínua
for i, (label, data) in enumerate(cdf_data_64qam):
    sorted_data = np.sort(data)
    cdf = np.linspace(0, 1, len(sorted_data))
    line, = ax.plot(sorted_data, cdf, linestyle='-', linewidth=1.5, color=cmap(i),
                    label=f'64-QAM - {label}')
    legend_handles.append(line)

# 256-QAM: curva + marcador 'o' a cada 10 pontos
for i, (label, data) in enumerate(cdf_data_256qam):
    sorted_data = np.sort(data)
    cdf = np.linspace(0, 1, len(sorted_data))

    # linha principal
    ax.plot(sorted_data, cdf, linestyle='--', linewidth=1.5, color=cmap(i))

    # marcador a cada 10 pontos
    marker_indices = np.arange(0, len(sorted_data), 10)
    ax.plot(sorted_data[marker_indices], cdf[marker_indices], linestyle='None',
            marker='*', markersize=6, color=cmap(i))

    # linha + marcador para legenda
    legend_marker = Line2D([], [], linestyle='--', marker='*', color=cmap(i),
                           markersize=6, linewidth=1.5, label=f'256-QAM - {label}')
    legend_handles.append(legend_marker)

# Finalização
ax.set_xlabel('Throughput (Mbps)', fontsize=fontSize)
ax.set_ylabel('CDF', fontsize=fontSize)
#ax.set_title('Throughput WiFi', fontsize=fontSize)
plt.ylim(0, 1)
ax.grid(True, linestyle='--', linewidth=0.7, alpha=0.6)
ax.legend(handles=legend_handles, fontsize=fontSizeLeg)
plt.tight_layout()

# Salvar
cdf_all_path = os.path.join(results_path, "cdf_tput_WiFi.png")
plt.savefig(cdf_all_path, dpi=600, bbox_inches='tight', transparent=True)
plt.close()

In [12]:
from matplotlib.lines import Line2D

# Armazenar dados (sem offset)
cdf_data_64qam = []
cdf_data_256qam = []

for user in users:
    # SISO
    key_64qam_siso = f'tput5g_64qam_siso_{user}user'
    key_256qam_siso = f'tput5g_256qam_siso_{user}user'

    # MIMO
    key_64qam_mimo = f'tput5g_64qam_mimo_{user}user'
    key_256qam_mimo = f'tput5g_256qam_mimo_{user}user'

    # Verificação
    keys_required = [key_64qam_siso, key_256qam_siso, key_64qam_mimo, key_256qam_mimo]
    if any(k not in medias_por_job for k in keys_required):
        print(f"Dados ausentes para {user} usuário(s). Pulando.")
        continue

    # Coleta dos dados
    y_siso_64 = medias_por_job[key_64qam_siso]['downlink_Mbps'].astype(float).values
    y_siso_256 = medias_por_job[key_256qam_siso]['downlink_Mbps'].astype(float).values
    y_mimo_64 = medias_por_job[key_64qam_mimo]['downlink_Mbps'].astype(float).values
    y_mimo_256 = medias_por_job[key_256qam_mimo]['downlink_Mbps'].astype(float).values

    # Armazenamento
    cdf_data_64qam.append((f'{user} User(s) - SISO', y_siso_64))
    cdf_data_64qam.append((f'{user} User(s) - MIMO', y_mimo_64))
    cdf_data_256qam.append((f'{user} User(s) - SISO', y_siso_256))
    cdf_data_256qam.append((f'{user} User(s) - MIMO', y_mimo_256))

# --- Plot CDF ---
fig, ax = plt.subplots(figsize=figsize)

legend_handles = []

# 64-QAM: curva contínua
for i, (label, data) in enumerate(cdf_data_64qam):
    sorted_data = np.sort(data)
    cdf = np.linspace(0, 1, len(sorted_data))
    line, = ax.plot(sorted_data, cdf, linestyle='-', linewidth=1.5, color=cmap(i),
                    label=f'64-QAM - {label}')
    legend_handles.append(line)

# 256-QAM: curva + marcador 'o' a cada 10 pontos
for i, (label, data) in enumerate(cdf_data_256qam):
    sorted_data = np.sort(data)
    cdf = np.linspace(0, 1, len(sorted_data))

    # linha principal
    ax.plot(sorted_data, cdf, linestyle='--', linewidth=1.5, color=cmap(i))

    # marcador a cada 10 pontos
    marker_indices = np.arange(0, len(sorted_data), 10)
    ax.plot(sorted_data[marker_indices], cdf[marker_indices], linestyle='None',
            marker='*', markersize=6, color=cmap(i))

    # linha + marcador para legenda
    legend_marker = Line2D([], [], linestyle='--', marker='*', color=cmap(i),
                           markersize=6, linewidth=1.5, label=f'256-QAM - {label}')
    legend_handles.append(legend_marker)

# Finalização
ax.set_xlabel('Throughput (Mbps)', fontsize=fontSize)
ax.set_ylabel('CDF', fontsize=fontSize)
#ax.set_title('Throughput 5G', fontsize=fontSize)
plt.xlim(right=125)
plt.ylim(0, 1)
ax.grid(True, linestyle='--', linewidth=0.7, alpha=0.6)
ax.legend(handles=legend_handles, fontsize=fontSizeLeg)
plt.tight_layout()

# Salvar
cdf_all_path = os.path.join(results_path, "cdf_tput_5g.png")
plt.savefig(cdf_all_path, dpi=600, bbox_inches='tight', transparent=True)
plt.close()